In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/12/25 15:37:54] INFO     Found credentials from IAM Role:                                   ]8;id=883895;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=969466;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'dlv2-parser'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# preprocessing.py
COPY preprocessing.py ${LAMBDA_TASK_ROOT}

# cls_parser.pkl
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# counters
COPY functions_counters.py ${LAMBDA_TASK_ROOT}

# counters pricing
COPY functions_counters_pricing.py ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Load files from Flask app

In [4]:
list_str_filenames = [
    'api.py',
    'cls_parser.pkl',
    'preprocessing.py',
    'requirements.txt',
    'functions_counters.py',
    'functions_counters_pricing.py',
]
for str_filename in list_str_filenames:
    str_source = f'../07_flask_app/app/{str_filename}'
    str_destination = f'./{str_filename}'
    shutil.copyfile(str_source, str_destination)

### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pickle
import pandas as pd 
import json
pd.options.mode.chained_assignment = None # suppress warning

# lambda handler
def lambda_handler(event, context):
    # import parser
    print('Loading parser...')
    print('')
    cls_parser = pickle.load(open('cls_parser.pkl', 'rb'))
    # get payload
    print('Getting request...')
    print('')
    try:
        dict_json_request = event['request']
    except KeyError:
        dict_json_request = event
    str_json_request = json.dumps(dict_json_request)
    
    # parse payload
    print('Parsing payload...')
    cls_parser.get_data(str_request=str_json_request)
    cls_parser.engineer_pmt_hx()
    cls_parser.preprocessing()
    cls_parser.get_predictions()
    #cls_parser.interpolate()
    cls_parser.adverse_action()
    #cls_parser.counter_offers()
    cls_parser.generate_response()
    # extract output
    print('Extracting output...')
    print('')
    dict_response = cls_parser.dict_response
    print(dict_response)
    # return output_final
    return dict_response

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=dlv2-parser

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 764B done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.8
#2 DONE 0.2s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/10] FROM public.ecr.aws/lambda/python:3.8@sha256:fa74502074c43e125ce6a1210b1a665c5e6144d4f9eae25b7a57f88b22dce4d8
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.22MB done
#5 DONE 0.0s

#6 [ 5/10] COPY api.py /var/task
#6 CACHED

#7 [ 7/10] COPY cls_parser.pkl /var/task
#7 CACHED

#8 [ 3/10] COPY requirements.txt  .
#8 CACHED

#9 [ 4/10] RUN  pip3 install -r requirements.txt --target "/var/task"
#9 CACHED

#10 [ 6/10] COPY preprocessing.py /var/task
#10 CACHED

#11 [ 9/10] COPY functions_counters_pricing.py /var/task
#11 CACHED

#12 [ 2/10] RUN pip install --upgrade pip
#12 CACHED

#13 [ 8/10] COPY functions_counters.py /var/task
#13 CACHE

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'dlv2-parser' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/dlv2-parser]
08f9d62bfbeb: Preparing
744fcf902292: Preparing
9ab59bb0cef6: Preparing
7b6b789dbe81: Preparing
dd14c98fbf59: Preparing
58acc74e3de5: Preparing
5b274ae51a5f: Preparing
8e691a79d19e: Preparing
cdb489c91da7: Preparing
58acc74e3de5: Waiting
ea4786659916: Preparing
5b274ae51a5f: Waiting
27f06e397921: Preparing
e1b8ef616f15: Preparing
797618fd6cee: Preparing
cdb489c91da7: Waiting
c62a7f74983e: Preparing
27f06e397921: Waiting
e1b8ef616f15: Waiting
ea4786659916: Waiting
797618fd6cee: Waiting
814345d22610: Preparing
c62a7f74983e: Waiting
814345d22610: Waiting
08f9d62bfbeb: Layer already exists
dd14c98fbf59: Layer already exists
9ab59bb0cef6: Layer already exists
744fcf902292: Layer already exists
7b6b789dbe81: Layer already exists
5b274ae51a5f: Layer already exists
8e691a79d19e: Layer already exists
58acc74e3de5: Layer already exists
ea4786659916: Layer already exists
cdb489c91da7: Layer already exists
e1b

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

[03/12/25 15:37:58] INFO     Found credentials from IAM Role:                                   ]8;id=75892;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=608281;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 12 Mar 2025 15:37:58 GMT',
                                      'x-amzn-requestid': '9047f168-c8fc-44b8-99ef-f1952e166b81'},
                      'HTTPStatusCode': 204,
                      'RequestId': '9047f168-c8fc-44b8-99ef-f1952e166b81',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=180, # 3 minutes
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '8e64d54fe36fc4839bad786e8c39c8e1d7baac50817d513b6f64e50497bbf8da',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:dlv2-parser',
 'FunctionName': 'dlv2-parser',
 'LastModified': '2025-03-12T15:37:58.617+0000',
 'LoggingConfig': {'LogFormat': 'Text', 'LogGroup': '/aws/lambda/dlv2-parser'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1167',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 12 Mar 2025 15:37:59 GMT',
                                      'x-amzn-requestid': 'f4ea7e81-5698-44d6-b39a-d122b5271e87'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'f4ea7e81-5698-44d6-b39a-d122b5271e87',
                      'RetryA

### Clean-up

In [11]:
list_str_filenames = [
    'requirements.txt',
    'preprocessing.py',
    'cls_parser.pkl',
    'Dockerfile',
    'api.py',
    'lambda_function.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
]
for str_file in list_str_filenames:
    try:
        os.remove(str_file)
    except:
        pass